# Process Pool Executor
The `ProcessPoolExecutor` class is the most modern and easiest way to convert a for-loop to run in parallel for CPU-bound tasks.

Process pools are a design pattern that let you execute and manage heterogeneous, discrete, and ad hoc tasks.

In [1]:
from random import random
from time import sleep
from multiprocessing import Process
from concurrent.futures import ProcessPoolExecutor, TimeoutError, wait, as_completed, FIRST_COMPLETED, FIRST_EXCEPTION

## Processes, Executors, and Process Pools
**1. What is a process-based concurrency**

A process is a computer program.

Every Python program is a process and has one thread called the main thread used to execute our program instructions. Each process is, in fact, one instance of a Python interpreter that executes Python bytecode called the `MainProcess`

**2. Process pools provide reusable workers**

A process pool is a programming pattern for automatically managing a pool of worker processes. Responsible for:
* It controls when they are created, such as when they are needed.
* It controls how many tasks each worker can execute before being replaced.
* It also controls what workers should do when they are not being used, such as making them wait without consuming computational resources.

The `concurrent.futures` module was introduced by Brian Quinlan.

The `ProcessPoolExecutor` extends the `Executor` class and will return `Future` objects when it is called.
* `Executor`: Parent class for the `ProcessPoolExecutor` that defines basic life-cycle operations for the pool.
* `Future`: Object returned when submitting tasks to the process pool that may complete later.

The `Executor` class defines three methods used to control our process pool:
* `submit()`: Dispatch a function to be executed and return a `Future` object. 
* `map()`: Call a function for each item in a iterable. Each application to the function to an element will happen concurrently instead of sequentially.
* `shutdown()`: Shut down the `Executor`.

A `Future` is an object that represents a delayed result for an asynchronous task. Sometimes called a `promise` or `delay`. It provides a context for the result of a task that may or may not be executing and a way of getting a result once it is available.

A `Future` object is returned from an `Executor` when calling the `submit()` method to dispatch a task to be executed asynchronously.

`Future` object methods for inspecting the status of the task:
* `cancelled()`: Returns `True` if the task was canceled before being executed. A running task cannot be canceled.  
* `running()`: Returns `True` if the task is currently running.
* `done()`: Returns `True` if the task has completed or was canceled.

`Future` object methods for accessing the `result()` method. Both allow a timeout to be specified as an argument. If the timeout expires, then a `TimeoutError` will be raised:
* `result()`: Access the result from running the task.
* `exception()`: Access any exception raised while running the task.

Automatically call a function once the task is completed:
* `add_done_callback()`: Add a callback function to the task to be executed by the process pool once the task is completed.

**3. How to use the `ProcessPoolExecutor`**

`ProcessPoolExecutor` three main steps in the life-cycle:
1. Create: Create the process pool by calling the constructor `ProcessPoolExecutor()`.
2. Execute: Run tasks using workers via the `map()` or `submit()` methods.   
3. Shut Down: Shut down the process pool by calling `shutdown()`.

## Configure the `ProcessPoolExecutor`
**1. Configure the `ProcessPoolExecutor`**

Arguments:
* `max_workers`: Maximum number of worker processes to use in the pool.
* `mp_context`: The multiprocessing context is used to create worker processes.
* `initializer`: Function executed after each worker process is created.
* `initargs`: Arguments to the worker process initialization function.

**2. Configure the number of worker processes.**

Default Worker Processes = min(61, Logical CPUs)

```python
# protect the entry point
if __name__ == '__main__':
    # create a process pool
    exe = ProcessPoolExecutor()
    # report the status of the process pool
    print(exe._max_workers)
    # shutdown the process pool
    exe.shutdown()
```
Configure it yourself:
```python
# create a process pool with 4 workers
exe = ProcessPoolExecutor(max_workers=4)
```
There is no upper limit on the number of processes that can be created on most platforms. The exception is Windows that limits the maximum number of processes to 61.

If we are expecting to perform computational work in the main process in addition to the multiprocessing pool, consider setting the number of processes in the pool to be equal to the number of logical CPUs in our system minus one, to allow the main process to execute.

If we have particularly CPU intensive tasks, consider configuring the number of processes to be equal to the number of physical CPUs instead of the number of logical CPUs.

In [ ]:
# custom task function executed in the process pool
def task(number):
    # block for a moment
    sleep(1)
    # report a message
    if number % 10 == 0:
        print(f"> task {number} done", flush=True)
        
# protect the entry point
if __name__ == "__main__":
    # create a process pool
    with ProcessPoolExecutor(50) as exe:
        # issue many tasks to the pool
        _ = exe.map(task, range(50))

**3. Configure the start method.**

Three start methods:
* `spawn`: start a new Python process. Default on Windows and MacOS.
* `fork`: copy a Python process from an existing process. Not supported in Windows.
* `forkserver`: new process from which future forked processes will be copied.

```python
# create a new context with the spawn start method
ctx = get_context('spawn')
# create a process pool with a given context
exe = ProcessPoolExecutor(mp_context=ctx)
```

**4. Configure the worker initializer function.**

May be helpful for worker processes to prepare resources that may be used across the execution of many tasks, such as logging infrastructure or a result queue.
```python
# create a process pool and initialize workers
exe = ProcessPoolExecutor(initializer=init, initargs=(a1, a2))
```

In [ ]:
# custom function to be executed in a worker process
def task(number):
    # report a message
    print(f"Worker task {number}...", flush=True)
    # block for a moment
    sleep(1)
    
# initialize a worker in a process pool
def init():
    # report a message
    print("Initializing worker ...", flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # create and configure the process pool
    with ProcessPoolExecutor(2, initializer=init) as exe:
        # issue tasks to the process pool
        _ = exe.map(task, range(4))

## Execute Multiple Tasks Concurrently
**1. What is the `map()` method**

The `map()` method will traverse the provided iterable and issue one task to the `ProcessPoolExecutor` for each item in the iterable. The `map()` method does not block, instead it returns an iterable of return values immediately. The iterable of return values can be traversed and will block until the return value for each task is yielded, in the order that the tasks were issued.

We may want to limit how we are willing to wait for each single task to complete and yield a result. This can be achieved by setting the `timeout` argument and specifying how long we are willing to wait in seconds. If the timeout elapses before the result is yielded, a `TimeoutError` is raised and may need to be handled.

`map()` also takes a `chunksize` argument that can be used to group the number of issued tasks into batches or chunks for transmission and execution by worker processes. This can offer a large speed-up depending on the types of tasks being executed as it reduces the number of internal tasks objects that need to be managed by the `ProcessPoolExecutor` and the frequency of data serialization and desearialization of data required for tasks sent via function arguments and returned via returned values.

**2. Issue many tasks with one argument.**

Same function many times with different arguments in an iterable and handle return value of a function.

In [ ]:
# custom function to be executed in a worker process
def task(number):
    # generate a random value between 0 and 1
    value = random()
    # report a message
    print(f"Task generated {value}", flush=True)
    # block for a moment to simulate work
    sleep(value)
    # return a new value
    return number + value

# protect the entry point
if __name__ == "__main__":
    # create a process pool
    with ProcessPoolExecutor(4) as exe:
        # issue tasks to execute concurrently
        for result in exe.map(task, range(10)):
            # report results
            print(result)    

**3. Issue many tasks with multiple arguments.**

In [ ]:
# custom function to be executed in a worker process
def task(number, value):
    # report a message
    print(f"Task using {value}", flush=True)
    # block for a moment to simulate work
    sleep(value)
    # return a new value
    return number + value

# protect the entry point
if __name__ == "__main__":
    # create the process pool
    with ProcessPoolExecutor(4) as exe:
        # prepare random numbers between 0 and 1
        values = [random() for _ in range(10)]
        # issue tasks to execute concurrently
        for result in exe.map(task, range(10), values):
            # report results
            print(result)

**4. Issue many tasks with no return values.**

Use iterable of arguments, but not traversing the iterable of return values. The `map()` method will return immediately and because we will not be traversing the iterable and blocking until each task completes, the caller will not block.

In [ ]:
# custom function to be executed in a worker process
def task(number):
    # generate a random value between 0 an 1
    value = random()
    # report a message
    print(f"Task generated {value}", flush=True)
    # block for a moment to simulate work
    sleep(value)
    
# protect the entry point
if __name__ == "__main__":
    # create the process pool
    with ProcessPoolExecutor(4) as exe:
        # issue tasks to execute concurrently
        _ = exe.map(task, range(10))
    # wait automatically for all tasks to finish

**5. Execute tasks with a timeout**

Waiting forever is not a good practice and instead we should use a timeout wherever possible.

In [ ]:
# custom function to be executed in a worker process
def task(number):
    # generate a random value between 0 and 1
    value = random()
    # report a message
    print(f"Task generated {value}", flush=True)
    # block for a moment to simulate work
    sleep(number + value)
    # return a new value
    return number + value

# protect the entry point
if __name__ == "__main__":
    # create the process pool
    with ProcessPoolExecutor(4) as exe:
        try:
            # issue tasks to execute concurrently
            for result in exe.map(task, range(10), timeout=2):
                # report results
                print(result)
        except TimeoutError:
            print("Gave up, took too long")
    # report that we will wait for the tasks to complete
    print("Waiting for tasks to compelte ...")            

**6. Execute tasks in chunks.**

You can try value of `chunksize` argument could be a an even division of function calls by the number of workers.
$$
\text{chunksize} = \frac{\text{FunctionCalls}}{\text{NumWorkers}}
$$
In practice, there are no good ways of estimating a good `chunksize` value. Instead, the `chunksize` argument should be tuned to the specifics of the task and the system on which the program is being executed using a process of trial and error.

In [ ]:
# custom function to be executed in a worker process
def task(number):
    return number*2

# protect the entry point 
if __name__ == "__main__":
    with ProcessPoolExecutor(4) as exe:
        # issue tasks to execute concurrently
        _ = exe.map(task, range(10000), chunksize=500)
    # wait for all tasks to complete
    print("All done")

## Execute One-Off Tasks Asynchronously
**1. What the `submit()` method is.**

It returns a `Future` object immediately. It is a promise to return the results from the task (if any) and provides a way to determine if a specific task has been completed or not.

**2. Execute a  task with one argument.**

Execute a function that takes one argument and returns a value asynchronously.

In [ ]:
# custom function to be executed in a worker process
def task(number):
    # generate a random value between 0 and 1
    value = random()
    # report a message
    print(f"Task generated {value}", flush=True)
    # block for a moment to simulate work
    sleep(value)
    # return a new value
    return number + value

# protect the entry point
if __name__ == "__main__":
    # create the process pool
    with ProcessPoolExecutor() as exe:
        # issue an asynchronous task
        future = exe.submit(task, 100)
        # get the result once the task completes
        result = future.result()
        print(result)

**3. Execute a task with multiple arguments.**

In [ ]:
# custom function to be executed in a worker process
def task(number, value):
    # report a message
    print(f"Task received {value}", flush=True)
    # block for a moment to simulate work
    sleep(value)
    # return a new value
    return number + value

# protect the entry point
if __name__ == "__main__":
    # create the process pool 
    with ProcessPoolExecutor() as exe:
        # issue an asynchronous task
        future = exe.submit(task, 100, random())
        # get the result once the task completes
        result = future.result()
        # report the result
        print(result)

**4. Execute a task with no arguments.**

In [ ]:
# custom function to be executed in a worker process
def task():
    # generate a random value between 0 and 1
    value = random()
    # report a message
    print(f"Task generated {value}", flush=True)
    # block for a moment to simulate work
    sleep(value)
    # return a new value
    return value

# protect the entry point
if __name__ == "__main__":
    # create the process pool
    with ProcessPoolExecutor() as exe:
        # issue an asynchronous task
        future = exe.submit(task)
        # get the result once the task completes
        result = future.result()
        # report the result
        print(result)

**4. Execute a task with no return value.**

Fire-and-forget manner.

In [ ]:
# custom function to be executed in a worker process
def task(number):
    # generate a random value between 0 and 1
    value = random()
    # report a message
    print(f"Task generated {value}", flush=True)
    # block for a moment to simulate work
    sleep(value)
    
# protect the entry point
if __name__ == "__main__":
    # create the process pool
    with ProcessPoolExecutor() as exe:
        # issue an asynchronous task
        _ = exe.submit(task, 100)
    # wait for all tasks to finish

A `Future` object is returned immediately and ignored. The main process is then free to continue on.

**5. Execute many one-off tasks.**

The list of `Future` objects is then traversed and the return values are then reported in the order that the tasks were issued. Simulate a call to the `map()` method.

In [ ]:
# custom function to be executed in a worker process
def task(number):
    # generate a random value between 0 and 1
    value = random()
    # report a message
    print(f"Task generated {value}", flush=True)
    # block for a moment to simulate work
    sleep(value)
    # return a new value
    return number + value

# protect the entry point
if __name__ == "__main__":
    # create the process pool
    with ProcessPoolExecutor() as ex:
        # issue many asynchronous tasks systematically
        futures = [ex.submit(task, i) for i in range(5)]
        # enumerate futures and report results
        for future in futures:
            print(future.result())

## Query Asynchronously Tasks
**1. What are `Future` objects**

A `Future` object represents a handle on a task that is executed asynchronously. It can exist in one of three states:
1. Scheduled (pre-running). Queued in the process pool for execution until a worker process becomes available to execute it. It can be cancelled.
2. Running. Cannot be cancelled.
3. Done(post-running). A canceled task will always be in the `done` state.

**2. Check the status of `Future` objects**

Methods:
* `running()`: Returns `True` if the task is currently running. `False` otherwise. 
* `done()`: Returns `True` if the task has completed, `False` otherwise.
* `cancelled()`: Returns `True` if the task was canceled, `False` otherwise.

In [ ]:
# task function executed in a worker process
def work():
    # block for a moment
    sleep(0.5)
    
# protect the entry point 
if __name__ == "__main__":
    # create a process pool
    with ProcessPoolExecutor() as exe:
        # start one task
        future = exe.submit(work)
        # confirm that the task is running
        running = future.running()
        done = future.done()
        print(f"Future running = {running}, done={done}")
        # wait for the task to complete
        future.result()
        # confirm that the task is done
        running = future.running()
        done = future.done()
        print(f"Future running={running}, done={done}")

**3. Get results from `Future` objects**

This returns the result from the return function of the task we executed or `None` if the function did not return a value.

In [ ]:
# task function executed in a worker process
def work():
    # block for a moment
    sleep(1)
    # return a message
    return "All Done"

# protect the entry point 
if __name__ == "__main__":
    # create a process pool
    with ProcessPoolExecutor() as exe:
        # start one task
        future = exe.submit(work)
        # get the result from the task, blocks
        result = future.result()
        # report the result
        print(f"Got Result: {result}")

Good practice to limit how long we are willing to wait for a result.

In [ ]:
# task function executed in a worker process
def work():
    # block for a moment
    sleep(1)
    # return a message
    return "All Done"

# protect the entry point
if __name__ == "__main__":
    # create a process pool
    with ProcessPoolExecutor() as exe:
        # start one task
        future = exe.submit(work)
        try:
            # get the result from the task, blocks 
            result = future.result(timeout=0.5)
            # report the result 
            print(f"Got result: {result}")
        except TimeoutError:
            print("Gave up waiting for a result")

**4. Cancel `Future` objects**

We can cancel a task that has not yet started running in the `ProcessPoolExecutor`. Recall that when we put tasks into the pool with the `submit()` method that the tasks are added to an internal queue of work from which worker process can remove the tasks and execute them.

In [ ]:
# custom function executed in a worker process
def work(sleep_time):
    # block for a moment
    sleep(sleep_time)
    
# protect the entry point 
if __name__ == "__main__":
    # create a process pool
    with ProcessPoolExecutor(1) as exe:
        # start a long running task
        future1 = exe.submit(work, 2)
        running = future1.running()
        print(f"First task running={running}")
        # start a second
        future2 = exe.submit(work, 0.1)
        running = future2.running()
        print(f"Second task running={running}")
        # cancel the second task
        canceled = future2.cancel()
        print(f"Second task was canceled: {canceled}") 

**5. Add callbakcs to `Future` objects**

Callback function to be called once the task has completed. A task is completed if it finishes normally, if it is canceled or if an exception is raised.

In [ ]:
# custom function to call when a task is completed
def custom_callback(future):
    # report a message
    print(f"Custom callback got: {future.result()}", flush=True)
    
# custom task function executed in a worker process
def work():
    # block for a moment
    sleep(1)
    # return a result
    return 99

# protect the entry point
if __name__ == "__main__":
    # create a process pool
    with ProcessPoolExecutor() as exe:
        # execute the task
        fut = exe.submit(work)
        # add the custom callback
        fut.add_done_callback(custom_callback)

**6. Get exceptions from `Future` objects**

If an exception is raised during the execution of the task, it will be raised again automatically when we attempt to retrieve the result from the `Future`.

As such, if an exception can reasonably be raised within the task, then we can handle it when retrieving the result.

In [ ]:
# custom task function executed by a worker process
def work():
    # block for a moment
    sleep(1)
    # raise an exception
    raise Exception("Something bad happened!")
    # never gets here
    
# protect the entry point
if __name__ == "__main__":
    # create a process pool
    with ProcessPoolExecutor() as exe:
        # execute our task
        future = exe.submit(work)
        # wait for the task to be done
        wait([future])
        # get the exception
        exception = future.exception()
        print(f"Task exception={exception}")
        # get the result from the task
        try:
            result = future.result()
        except Exception:
            print("Unable to get the result")

We can also access an exception raised in the task via the `exception()` method.

## Manage Collections of Asynchronously Tasks
**1. What are the module functions**
The `concurrent.futures` module provides two module utility functions for working with collections of `Future` objects:
* `as_completed()`: Yield `Future` objects in the order that tasks are completed.
* `wait()`: Wait for a condition in a collection of `Future` objects.

Both:
* Help us wait for tasks to be done.
* Can help us wait for all tasks to be done, or the first task to be done.
* Require us to have called `submit()` to create `Future` objects.
* Functions are optional, e.g., we don't have to use them when executing asynchronous tasks.

We should use the `as_completed()` function when we need task results in the order that tasks are completed.

We should use the `wait()` function when we need to wait on the first or all tasks to complete or on the first task to raise an unhandled error or exception.

**2. Handle results as tasks finish**

The `as_completed()` module function takes a collection of `Future` objects and returns immediately with an iterable of the provided `Future` objects.

Importantly, the iterable will yield `Future` objects in the order that their associated tasks are completed.

It is good practice for a program to not potentially wait forever. As such `as_completed()` function takes a `timeout` argument that limits how long the caller is willing to wait to yield the next `Future` object from the iterable. If the caller waits longer than the timeout number of seconds a `TimeoutError` will be raised that may need to be handled.
```python
try:
    # handle results in completion order with timeout 
    for future in as_completed(futures, timeout=5):
        # get the result
        result = future.result()
        # report the result
        print(result)
except TimeoutError:
    # wait too long
```

In [ ]:
# custom function executed by a worker process
def task(number):
    # block for a fraction of a second
    sleep(random())
    # return task number
    return number

# protect the entry point
if __name__ == "__main__":
    # start the process pool
    with ProcessPoolExecutor(10) as exe:
        # submit tasks and collect futures 
        futs = [exe.submit(task, i) for i in range(10)]
        # handle task results as they are completed
        for future in as_completed(futs):
            # retrieve the result
            result = future.result()
            # report the result
            print(f"> result for task {result}")

**3. Wait for all tasks**

The `wait` module function takes a collection of `Future` objects and will block until a condition is met with regard to the `Future` objects.

The wait condition is specified via the `return_when` argument and may be one of:
* `FIRST_COMPLETED`: Block until any one task is completed.
* `FIRST_EXCEPTION`: Block until any one task raises an unhandled exception.
* `ALL_COMPLETED`: Block until all tasks are completed.

Recall a completed task means that the target function exited normally, raised and exception, or was canceled via the `Future` object.

The `wait()` module function returns a named tuple containing two sets, `done` and `not_done`.

The first set contains those `Future` objects that meet the condition specified via the `return_when` argument and are `done` (completed), and the second set contains all other `Future` objects that are not `done`.

By default, the `wait()` function will wait for all provided tasks to be completed.
```python
# wait for all task to complete
done, not_done = wait(futures)
```
```python
# wait for all task to complete
done, not_done = wait(futures, return_when=ALL_COMPLETED)
```
We may also specify a `timeout` argument that limits how long the caller is willing to block in seconds.
```python
# wait for all task to complete with timeout
done, not_done = wait(futures, timeout=5)
```

In [ ]:
# custom task function executed by a worker process
def task(number):
    # block for a fraction of a second
    sleep(random())
    # report value
    print(number, flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # start the process pool
    with ProcessPoolExecutor(10) as exe:
        # issue all tasks and gather the futures
        futs = [exe.submit(task, i) for i in range(10)]
        # wait for all tasks to complete
        _ = wait(futs)
        # report a message
        print("All tasks are done!")

**4. Wait for the first task**

We can call the `pop()` method on the `done` set to retrieve the first and only task that was completed.

```python
# get the first task to complete
future = done.pop()
```

In [ ]:
# custom task function executed by a worker process
def task(number):
    # block for a fraction of a second
    sleep(random())
    # report that the task completed
    print(f">task {number} is done", flush=True)
    # return the task number
    return number

# protect the entry point
if __name__ == "__main__":
    # start the process pool
    with ProcessPoolExecutor(10) as exe:
        # submit tasks and collect futures
        futs = [exe.submit(task, i) for i in range(10)]
        # wait until any task completes
        done, _ = wait(futs, return_when=FIRST_COMPLETED)
        # get the first task to complete
        future = done.pop()
        # get the result from the first task to complete
        result = future.result()
        # report the result
        print(f"Main first result: {result}")

**5. Wait for the first task failure**

Fail with an `Error` or `Exception`.

If none of the provided tasks fail, then the call will block until all tasks are completed and the `done` set will contain all `Future` objects.

Otherwise if at least on task fails with an exception, the `done` set will contain the failed tasks. 

We can call the `pop()` method on the `done` set to retrieve the `Future` that failed.

In [ ]:
# custom task function executed by a worker process
def task(number):
    # generate a random number between 0 and 1
    value = random()
    # block for a fraction of a second
    sleep(value)
    # check for a failure
    if value < 0.5:
        raise Exception(f"Task {number} Failed")
    # report that the task completed
    print(f">task {number} is done", flush=True)
    # return the task number
    return number

# protect the entry point 
if __name__ == "__main__":
    # start the process pool
    with ProcessPoolExecutor(10) as exe:
        # submit task and collect futures
        futs = [exe.submit(task, i) for i in range(10)]
        # wait until any task completes
        done, _ = wait(futs, return_when=FIRST_EXCEPTION)
        # check that no tasks failed
        if len(done) == len(futs):
            print("No tasks failed")
        else:
            # get the first task to complete
            future = done.pop()
            # report the exception
            print(f"First fail: {future.exception()}")

## Case Study: Calculate Fibonacci Numbers
The sequence of numbers are related to the golden ratio and are named afer the discoverer of the sequence known as Fibonacci.

The numbers in the sequence are calculated as the sum of the last two numbers in the sequence where the first two numbers in the sequence are 0 and 1.

$$f_n = f_{n-1} + f_{n-2}$$

**Slow**
1. *Recursively*: Writing recursive functions is a common exercise for new programmers and computer science students, although it is probably poor form in modern software development.

In [3]:
# calculate the nth fibonacci number
def fibonacci(n):
    # check the start of the sequence
    if n <= 1:
        return n
    return (fibonacci(n-1) + fibonacci(n-2))

# protect the entry point
if __name__ == "__main__":
    # calculate some fibonacci numbers
    for i in range(30):
        print(f"f{i} = {fibonacci(i)}")

f0 = 0
f1 = 1
f2 = 1
f3 = 2
f4 = 3
f5 = 5
f6 = 8
f7 = 13
f8 = 21
f9 = 34
f10 = 55
f11 = 89
f12 = 144
f13 = 233
f14 = 377
f15 = 610
f16 = 987
f17 = 1597
f18 = 2584
f19 = 4181
f20 = 6765
f21 = 10946
f22 = 17711
f23 = 28657
f24 = 46368
f25 = 75025
f26 = 121393
f27 = 196418
f28 = 317811
f29 = 514229


By the time we start calculating numbers around 30, this recursive approach becomes very slow.

2. *Iteratively*

In [6]:
# calculate the nth fibonacci number
def fibonacci(n):
    # start the sequence
    f0, f1 = 0, 1
    # compute the nth number
    for _ in range(0, n):
        f0, f1 = f1, (f1 + f0)
    return f0

# protect the entry point
if __name__ == "__main__":
    # calculate some fibonacci numbers
    for i in range(30):
        print(f"f({i}) = {fibonacci(i)}")

f(0) = 0
f(1) = 1
f(2) = 1
f(3) = 2
f(4) = 3
f(5) = 5
f(6) = 8
f(7) = 13
f(8) = 21
f(9) = 34
f(10) = 55
f(11) = 89
f(12) = 144
f(13) = 233
f(14) = 377
f(15) = 610
f(16) = 987
f(17) = 1597
f(18) = 2584
f(19) = 4181
f(20) = 6765
f(21) = 10946
f(22) = 17711
f(23) = 28657
f(24) = 46368
f(25) = 75025
f(26) = 121393
f(27) = 196418
f(28) = 317811
f(29) = 514229


3. *Many iteratively*

In [9]:
# calculate the nth fibonacci number
def fibonacci(n):
    # start the sequence
    f0, f1 = 0, 1
    # compute the nth number
    for _ in range(0, n):
        f0, f1 = f1, (f1 + f0)
    return f0

# protect the entry point
if __name__ == "__main__":
    # fibonacci numbers to calculate
    numbers = range(10_000)
    # calculate fibonnacci numbers
    fibs = map(fibonacci, numbers)
    # store results
    results = dict(zip(numbers, fibs))
    print("Done")

Done


**Fast**

In [ ]:
# calculate the nth fibonacci number
def fibonacci(n):
    # start the sequence
    f0, f1 = 0, 1
    # compute the nth number
    for _ in range(0, n):
        f0, f1 = f1, (f1 + f0)
    return f0

# protect the entry point
if __name__ == "__main__":
    # create the process pool
    with ProcessPoolExecutor() as exe:
        # fibonacci numbers to calculate
        numbers = range(10_000)
        # calculate concurrently
        fibs = exe.map(fibonacci, numbers)
        # store the results
        results = dict(zip(numbers, fibs))
    print("Done")

**Faster with chunksize**

A good starting point for setting the chunksize is to divide the number of tasks by the number of logical CPUs. This is a good idea if all tasks have about the same execution duration, but that is not the case here as calculating the `n=10` Fibonacci number is a lot faster than calculating the f=1000 number. A such, it is a good idea to try different chunksize values and see what works best.

In [ ]:
# calculate the nth fibonacci number
def fibonacci(n):
    # start the sequence
    f0, f1 = 0, 1
    # compute the nth number
    for _ in range(0, n):
        f0, f1 = f1, (f1 + f0)
    return f0

# protect the entry point
if __name__ == "__main__":
    # create the process pool
    with ProcessPoolExecutor() as exe:
        # fibonacci numbers to calculate
        numbers = range(10_000)
        # calculate concurrently
        fibs = exe.map(fibonacci, numbers, chunksize=50)
        # store the results
        results = dict(zip(numbers, fibs))
    print("Done")